In [1]:
# =============================================================================
# CELLULE 1: Installation des dependances
# =============================================================================
# GLiNER pour le modele NER, scikit-learn pour le split des donnees

%pip install -q gliner scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# =============================================================================
# CELLULE 2: Imports et configuration de l'environnement
# =============================================================================

import json
import random
from typing import Dict, List, Any
from sklearn.model_selection import train_test_split
from gliner import GLiNER

# Configuration de la reproductibilite
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Verification GPU
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Appareil utilise: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Appareil utilise: cuda
GPU: NVIDIA GeForce RTX 5090


/home/hounfodjidagba/miniconda3/envs/ner/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =============================================================================
# CELLULE 3: Chargement et exploration du dataset
# =============================================================================

def charger_dataset(chemin_fichier: str) -> List[Dict]:
    """
    Charge le dataset JSON contenant les annotations NER.
    
    Args:
        chemin_fichier: Chemin vers le fichier JSON
        
    Returns:
        Liste des exemples annotes
    """
    with open(chemin_fichier, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

# Charger les donnees (ajuster le chemin pour Colab si necessaire)
chemin_dataset = 'travel_ner_final.json'
dataset = charger_dataset(chemin_dataset)

# Exploration du dataset
print(f"Nombre total d'exemples: {len(dataset)}")
print(f"\nExemple de donnee:")
print(json.dumps(dataset[0], indent=2, ensure_ascii=False))

# Statistiques sur les entites
stats_labels = {"Departure": 0, "Destination": 0}
for exemple in dataset:
    for entite in exemple.get("entities", []):
        label = entite.get("label")
        if label in stats_labels:
            stats_labels[label] += 1

print(f"\nDistribution des entites:")
for label, count in stats_labels.items():
    print(f"  {label}: {count}")

Nombre total d'exemples: 961

Exemple de donnee:
{
  "sentence": "Je veux un vol de Marseille à Lyon",
  "entities": [
    {
      "text": "Marseille",
      "label": "Departure",
      "start": 18,
      "end": 27
    },
    {
      "text": "Lyon",
      "label": "Destination",
      "start": 30,
      "end": 34
    }
  ]
}

Distribution des entites:
  Departure: 527
  Destination: 889


In [4]:
# =============================================================================
# CELLULE 4: Conversion du format vers le format GLiNER pour fine-tuning
# =============================================================================

# Labels d'entites a entrainer
ENTITY_LABELS = ["Departure", "Destination"]

def convertir_vers_format_gliner(donnees: List[Dict], labels: List[str]) -> List[Dict]:
    """
    Convertit le dataset vers le format attendu par GLiNER pour le fine-tuning.
    
    Format GLiNER attendu:
        {
            "tokenized_text": ["token1", "token2", ...],
            "ner": [[start, end, label], ...],
            "label": ["Departure", "Destination"]  # OBLIGATOIRE
        }
    
    Args:
        donnees: Liste des exemples au format original
        labels: Liste des types d'entites a entrainer
        
    Returns:
        Liste des exemples au format GLiNER
    """
    dataset_gliner = []
    
    for exemple in donnees:
        sentence = exemple["sentence"]
        tokens = sentence.split()  # Tokenisation simple par espaces
        
        # Convertir les entites avec leurs positions en tokens
        ner_annotations = []
        
        for entite in exemple.get("entities", []):
            entity_text = entite["text"]
            label = entite["label"]
            
            # Trouver la position du token de debut et fin
            # On reconstruit la position caractere par caractere
            char_pos = 0
            start_token = None
            end_token = None
            
            for i, token in enumerate(tokens):
                if char_pos == entite["start"]:
                    start_token = i
                if char_pos + len(token) == entite["end"]:
                    end_token = i + 1
                    break
                char_pos += len(token) + 1  # +1 pour l'espace
            
            if start_token is not None and end_token is not None:
                ner_annotations.append([start_token, end_token, label])
        
        dataset_gliner.append({
            "tokenized_text": tokens,
            "ner": ner_annotations,
            "label": labels  # CHAMP OBLIGATOIRE pour GLiNER
        })
    
    return dataset_gliner

# Conversion du dataset avec les labels
dataset_gliner = convertir_vers_format_gliner(dataset, ENTITY_LABELS)

# Verification de la conversion
print("Exemple apres conversion au format GLiNER:")
print(json.dumps(dataset_gliner[0], indent=2, ensure_ascii=False))
print(f"\nPhrase originale: {dataset[0]['sentence']}")

Exemple apres conversion au format GLiNER:
{
  "tokenized_text": [
    "Je",
    "veux",
    "un",
    "vol",
    "de",
    "Marseille",
    "à",
    "Lyon"
  ],
  "ner": [
    [
      5,
      6,
      "Departure"
    ],
    [
      7,
      8,
      "Destination"
    ]
  ],
  "label": [
    "Departure",
    "Destination"
  ]
}

Phrase originale: Je veux un vol de Marseille à Lyon


In [5]:
# =============================================================================
# CELLULE 5: Division du dataset en train/validation
# =============================================================================

# Configuration du split
VALIDATION_RATIO = 0.15  # 15% pour la validation

# Division aleatoire
train_data, val_data = train_test_split(
    dataset_gliner,
    test_size=VALIDATION_RATIO,
    random_state=RANDOM_SEED,
    shuffle=True
)

print(f"Taille ensemble d'entrainement: {len(train_data)} exemples")
print(f"Taille ensemble de validation: {len(val_data)} exemples")

# Afficher un exemple de chaque ensemble
print(f"\nExemple d'entrainement:")
print(f"  Tokens: {train_data[0]['tokenized_text']}")
print(f"  NER: {train_data[0]['ner']}")

Taille ensemble d'entrainement: 816 exemples
Taille ensemble de validation: 145 exemples

Exemple d'entrainement:
  Tokens: ['Vol', 'direct', 'Alger-Tunis']
  NER: []


In [6]:
# =============================================================================
# CELLULE 6: Configuration et lancement de l'entrainement
# =============================================================================

from gliner import GLiNERConfig
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding
from gliner.data_processing import WordsSplitter, GLiNERDataset
import os

# Charger le modele de base pre-entraine
MODEL_NAME = "urchade/gliner_multi-v2.1"  # Modele multilingue (supporte le francais)
print(f"Chargement du modele de base: {MODEL_NAME}")
model = GLiNER.from_pretrained(MODEL_NAME)

# Les labels a entrainer
LABELS = ["Departure", "Destination"]

# Configuration des arguments d'entrainement
training_args = TrainingArguments(
    output_dir="./travel_ner_model",
    learning_rate=5e-6,
    weight_decay=0.01,
    others_lr=1e-5,
    others_weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    dataloader_num_workers=0,
    use_cpu=not torch.cuda.is_available(),
    report_to="none",
    fp16=torch.cuda.is_available(),
)

print("\nConfiguration de l'entrainement:")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  FP16: {training_args.fp16}")

Chargement du modele de base: urchade/gliner_multi-v2.1


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 73071.50it/s]
/home/hounfodjidagba/miniconda3/envs/ner/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/hounfodjidagba/miniconda3/envs/ner/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  


Configuration de l'entrainement:
  Learning rate: 5e-06
  Batch size: 8
  Epochs: 15
  FP16: True


/home/hounfodjidagba/miniconda3/envs/ner/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [7]:
# =============================================================================
# CELLULE 7: Preparation des datasets et entrainement
# =============================================================================

# Creer les datasets pour GLiNER
train_dataset = GLiNERDataset(train_data, model.config, data_processor=model.data_processor)
val_dataset = GLiNERDataset(val_data, model.config, data_processor=model.data_processor)

# Data collator pour le padding
data_collator = DataCollatorWithPadding(model.config)

# Creer le trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_collator=data_collator,
)

# Lancer l'entrainement
print("=" * 60)
print("Debut de l'entrainement...")
print("=" * 60)

trainer.train()

print("=" * 60)
print("Entrainement termine!")
print("=" * 60)

100%|██████████| 816/816 [00:00<00:00, 660724.34it/s]


Total number of entity classes:  2


100%|██████████| 145/145 [00:00<00:00, 733623.74it/s]
/home/hounfodjidagba/miniconda3/envs/ner/lib/python3.11/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Total number of entity classes:  2
Debut de l'entrainement...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Epoch,Training Loss,Validation Loss
1,No log,9.971286
2,No log,1.580583
3,No log,0.414435
4,No log,0.174679
5,8.628200,0.034184
6,8.628200,0.017234
7,8.628200,0.092556
8,8.628200,0.000069
9,8.628200,0.000019
10,0.115000,0.000007


There were missing keys in the checkpoint model loaded: ['model.token_rep_layer.bert_layer.model.embeddings.word_embeddings.weight', 'model.token_rep_layer.bert_layer.model.embeddings.LayerNorm.weight', 'model.token_rep_layer.bert_layer.model.embeddings.LayerNorm.bias', 'model.token_rep_layer.bert_layer.model.encoder.layer.0.attention.self.query_proj.weight', 'model.token_rep_layer.bert_layer.model.encoder.layer.0.attention.self.query_proj.bias', 'model.token_rep_layer.bert_layer.model.encoder.layer.0.attention.self.key_proj.weight', 'model.token_rep_layer.bert_layer.model.encoder.layer.0.attention.self.key_proj.bias', 'model.token_rep_layer.bert_layer.model.encoder.layer.0.attention.self.value_proj.weight', 'model.token_rep_layer.bert_layer.model.encoder.layer.0.attention.self.value_proj.bias', 'model.token_rep_layer.bert_layer.model.encoder.layer.0.attention.output.dense.weight', 'model.token_rep_layer.bert_layer.model.encoder.layer.0.attention.output.dense.bias', 'model.token_rep_la

Entrainement termine!


In [8]:
# =============================================================================
# CELLULE 8: Evaluation du modele (Precision, Recall, F1)
# =============================================================================

from collections import defaultdict

def evaluer_modele(model: GLiNER, donnees_test: List[Dict], labels: List[str]) -> Dict[str, Any]:
    """
    Evalue le modele sur un ensemble de test.
    Calcule Precision, Recall et F1 pour chaque type d'entite.
    
    Args:
        model: Modele GLiNER entraine
        donnees_test: Liste des exemples de test au format GLiNER
        labels: Liste des labels a evaluer
        
    Returns:
        Dictionnaire contenant les metriques
    """
    # Compteurs
    true_positives = defaultdict(int)
    false_positives = defaultdict(int)
    false_negatives = defaultdict(int)
    
    for exemple in donnees_test:
        tokens = exemple["tokenized_text"]
        text = " ".join(tokens)
        
        # Entites reelles (ground truth)
        entites_reelles = defaultdict(set)
        for start, end, label in exemple["ner"]:
            entity_text = " ".join(tokens[start:end])
            entites_reelles[label].add(entity_text.lower())
        
        # Prediction du modele
        predictions = model.predict_entities(text, labels, threshold=0.5)
        entites_predites = defaultdict(set)
        for pred in predictions:
            entites_predites[pred["label"]].add(pred["text"].lower())
        
        # Comparaison pour chaque label
        for label in labels:
            reelles = entites_reelles.get(label, set())
            predites = entites_predites.get(label, set())
            
            # True positives
            tp = reelles & predites
            true_positives[label] += len(tp)
            
            # False positives
            fp = predites - reelles
            false_positives[label] += len(fp)
            
            # False negatives
            fn = reelles - predites
            false_negatives[label] += len(fn)
    
    # Calcul des metriques
    metriques = {}
    
    for label in labels:
        tp = true_positives[label]
        fp = false_positives[label]
        fn = false_negatives[label]
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        metriques[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": tp + fn
        }
    
    # Macro average
    precision_macro = sum(m["precision"] for m in metriques.values()) / len(labels)
    recall_macro = sum(m["recall"] for m in metriques.values()) / len(labels)
    f1_macro = sum(m["f1"] for m in metriques.values()) / len(labels)
    
    metriques["macro_avg"] = {
        "precision": precision_macro,
        "recall": recall_macro,
        "f1": f1_macro
    }
    
    return metriques

# Evaluation sur l'ensemble de validation
print("=" * 60)
print("Evaluation du modele fine-tune")
print("=" * 60)

metriques = evaluer_modele(model, val_data, LABELS)

# Affichage des resultats
print(f"\n{'Label':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 60)

for label in LABELS:
    m = metriques[label]
    print(f"{label:<15} {m['precision']:<12.4f} {m['recall']:<12.4f} {m['f1']:<12.4f} {m['support']:<10}")

print("-" * 60)
m = metriques["macro_avg"]
print(f"{'Macro Avg':<15} {m['precision']:<12.4f} {m['recall']:<12.4f} {m['f1']:<12.4f}")

Evaluation du modele fine-tune

Label           Precision    Recall       F1-Score     Support   
------------------------------------------------------------
Departure       0.0000       0.0000       0.0000       52        
Destination     0.0000       0.0000       0.0000       89        
------------------------------------------------------------
Macro Avg       0.0000       0.0000       0.0000      


In [9]:
# =============================================================================
# CELLULE 9: Sauvegarde du modele fine-tune
# =============================================================================

# Chemin de sauvegarde
SAVE_PATH = "./travel_ner_gliner_finetuned"

# Sauvegarder le modele
model.save_pretrained(SAVE_PATH)

print(f"Modele sauvegarde dans: {SAVE_PATH}")
print(f"\nFichiers sauvegardes:")
for f in os.listdir(SAVE_PATH):
    print(f"  - {f}")

# Instructions pour telecharger depuis Colab
print("\n" + "=" * 60)
print("Pour telecharger le modele depuis Colab:")
print("=" * 60)
print("""
# Compresser le modele
!zip -r travel_ner_model.zip {SAVE_PATH}

# Telecharger
from google.colab import files
files.download('travel_ner_model.zip')
""")

Modele sauvegarde dans: ./travel_ner_gliner_finetuned

Fichiers sauvegardes:
  - spm.model
  - special_tokens_map.json
  - tokenizer.json
  - tokenizer_config.json
  - pytorch_model.bin
  - added_tokens.json
  - gliner_config.json

Pour telecharger le modele depuis Colab:

# Compresser le modele
!zip -r travel_ner_model.zip {SAVE_PATH}

# Telecharger
from google.colab import files
files.download('travel_ner_model.zip')



In [10]:
# =============================================================================
# CELLULE 10: Fonction d'inference pour de nouvelles phrases
# =============================================================================

def extraire_entites_voyage(phrase: str, model: GLiNER = None, chemin_modele: str = None) -> Dict[str, str]:
    """
    Extrait les entites Departure et Destination d'une phrase de voyage.
    
    Args:
        phrase: Phrase en francais decrivant un trajet
        model: Modele GLiNER (charge automatiquement si None)
        chemin_modele: Chemin vers le modele sauvegarde
        
    Returns:
        Dictionnaire au format {'Departure': 'ville', 'Destination': 'ville'}
    """
    # Charger le modele si non fourni
    if model is None:
        if chemin_modele is None:
            chemin_modele = "./travel_ner_gliner_finetuned"
        model = GLiNER.from_pretrained(chemin_modele)
    
    # Labels a extraire
    labels = ["Departure", "Destination"]
    
    # Extraction
    predictions = model.predict_entities(phrase, labels, threshold=0.5)
    
    # Formater la sortie
    output = {"Departure": None, "Destination": None}
    
    for pred in predictions:
        label = pred["label"]
        if label in output and output[label] is None:
            output[label] = pred["text"]
    
    return output

# ===== EXEMPLES D'UTILISATION =====
print("=" * 60)
print("Exemples d'inference avec le modele fine-tune")
print("=" * 60)

phrases_test = [
    "je voudrais partir à Dassa au départ de Cotonou",
    "Réservez-moi un vol de Marseille vers Lyon",
    "Comment aller de Dakar à Abidjan ?",
    "Je cherche un billet Toulouse Douala",
    "Vol direct Nantes Casablanca disponible ?",
    "Je veux partir de Paris demain matin",
    "Quel est le prix pour aller à Parakou ?"
]

for phrase in phrases_test:
    resultat = extraire_entites_voyage(phrase, model=model)
    print(f"\nPhrase: {phrase}")
    print(f"Resultat: {resultat}")

Exemples d'inference avec le modele fine-tune

Phrase: je voudrais partir à Dassa au départ de Cotonou
Resultat: {'Departure': None, 'Destination': 'Dassa au'}

Phrase: Réservez-moi un vol de Marseille vers Lyon
Resultat: {'Departure': 'Marseille vers', 'Destination': None}

Phrase: Comment aller de Dakar à Abidjan ?
Resultat: {'Departure': 'Dakar à', 'Destination': 'Abidjan ?'}

Phrase: Je cherche un billet Toulouse Douala
Resultat: {'Departure': 'Toulouse Douala', 'Destination': None}

Phrase: Vol direct Nantes Casablanca disponible ?
Resultat: {'Departure': 'Nantes Casablanca', 'Destination': None}

Phrase: Je veux partir de Paris demain matin
Resultat: {'Departure': 'Paris demain', 'Destination': None}

Phrase: Quel est le prix pour aller à Parakou ?
Resultat: {'Departure': None, 'Destination': 'Parakou ?'}
